<a href="https://colab.research.google.com/github/Dorthi12/SIH-26/blob/main/crop_recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌱 MODEL 2 — CROP RECOMMENDATION & SUITABILITY ENGINE

## 1. Objective

Model 2 is responsible for recommending and ranking suitable crops for a given agricultural location and season.

The system takes historical agricultural production records and learns which crops have historically performed best under a particular:

- State
- District
- Season
- Year

The final recommendation system should produce a ranked list such as:

1. Crop A — High suitability
2. Crop B — High suitability
3. Crop C — Moderate suitability
4. Crop D — Moderate suitability
5. Crop E — Lower suitability

The recommendation system is not intended to provide a single absolute agricultural decision. It provides ranked candidate crops that can later be combined with:

- Model 1 — Yield Prediction
- RAG knowledge base
- Weather/environment information when available
- Market information when available
- Agentic reasoning

---

# 2. Current Dataset

The cleaned agricultural dataset is:

`apy_clean.csv`

Current fields:

- `state`
- `district`
- `crop`
- `crop_year`
- `season`
- `area`
- `production`
- `yield`
- `zero_production_flag`

Historical coverage:

- Years: 1997–2020
- Multiple Indian states
- Multiple districts
- Multiple crops
- Multiple agricultural seasons

---

# 3. Training Data Required

Model 2 requires historical crop-level records containing:

| Field | Required | Purpose |
|---|---|---|
| state | YES | Geographic context |
| district | YES | Local geographic context |
| crop | YES | Candidate/target crop |
| crop_year | YES | Temporal context |
| season | YES | Seasonal context |
| area | OPTIONAL | Historical production context |
| production | NO as model input | Used only for data analysis |
| yield | YES | Used to determine historical crop performance |
| zero_production_flag | NO as model input | Used to exclude zero-production observations |

---

# 4. Important Data Leakage Rules

Production must NOT be supplied as a Model 2 prediction feature.

Production is directly related to yield:

`yield = production / area`

Therefore:

`production → forbidden model feature`

The recommendation model must not learn the answer from production.

Similarly, `yield` is not directly supplied as an input to the classifier.

Instead, historical yield is used to construct the recommendation target.

---

# 5. Recommendation Target Construction

The raw dataset does not contain a column called:

`recommended_crop`

Therefore, the target is derived from historical performance.

For every:

`state + district + season + crop_year`

combination:

1. Ignore zero-production observations.
2. Consider all crops available for that location and season.
3. Identify the crop with the highest historical yield.
4. Store that crop as the historical best-performing crop.

This produces a supervised learning target:

### Input

`state + district + season + crop_year`

### Target

`best-performing crop`

---

# 6. Model Features

The initial recommendation classifier uses:

- `state`
- `district`
- `season`
- `crop_year`

The following are NOT used as classifier inputs:

- `production`
- `yield`
- `zero_production_flag`

`area` is initially excluded because historical planted area largely reflects previous farmer decisions rather than independent crop suitability.

---

# 7. Temporal Validation Strategy

Random train/test splitting is prohibited.

Agricultural recommendation must respect time.

The split is:

### Training

1997–2017

### Validation

2018

### Test

2019

### Excluded

2020

This prevents future agricultural information from leaking into training.

---

# 8. Model Architecture

The first implementation uses:

`CatBoostClassifier`

CatBoost is selected because the dataset contains high-cardinality categorical variables such as:

- district
- crop
- state

Categorical features are handled natively.

---

# 9. Evaluation

The recommendation model will be evaluated using:

### Top-1 Accuracy

Whether the highest-probability recommendation is correct.

### Top-3 Accuracy

Whether the actual best historical crop appears in the top three recommendations.

### Top-5 Accuracy

Whether the actual best historical crop appears in the top five recommendations.

Top-K metrics are particularly important because the final agricultural system should provide multiple candidate crops rather than a single rigid answer.

---

# 10. Final Recommendation Pipeline

The final backend recommendation flow is:

User request
↓
State
District
Season
Year
↓
Model 2
↓
Candidate crop probabilities
↓
Top candidate crops
↓
Model 1 Yield Prediction
↓
Predicted yield for each candidate
↓
Ranking
↓
Top-N crop recommendations
↓
RAG / Agentic reasoning
↓
Final agricultural recommendation report

---

# 11. Backend Database Requirements

When the model is connected to the backend database, the database should maintain historical agricultural records containing at minimum:

- state
- district
- crop
- crop_year
- season
- area
- production
- yield

The model-training pipeline can periodically read these records from the database.

The backend should NOT send raw production or yield to the recommendation inference endpoint as user-input features.

---

# 12. Backend Inference Input

For a recommendation request, the backend should provide:

```json
{
    "state": "Maharashtra",
    "district": "Pune",
    "season": "Kharif",
    "year": 2025
}

backend rec. output

{
    "state": "Maharashtra",
    "district": "Pune",
    "season": "Kharif",
    "year": 2025,
    "recommendations": [
        {
            "crop": "Groundnut",
            "recommendation_probability": 0.82,
            "predicted_yield": 2.91,
            "rank": 1
        },
        {
            "crop": "Soybean",
            "recommendation_probability": 0.76,
            "predicted_yield": 2.64,
            "rank": 2
        }
    ]
}

02_crop_recommendation.ipynb
│
├── 1. Environment setup

├── 2. Load apy_clean.csv

├── 3. Dataset validation  

├── 4. Build recommendation target

├── 5. Temporal split

├── 6. Target/class coverage

├── 7. Train Model 2

├── 8. Evaluate Top-1 / Top-3 / Top-5

├── 9. Recommendation ranking

├── 10. Error analysis

├── 11. Feature importance

└── 12. Save model + metadata

In [1]:
# ============================================================
# MODEL 2 — CROP RECOMMENDATION
# ENVIRONMENT SETUP
# ============================================================

!pip -q install catboost

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    top_k_accuracy_score,
    classification_report,
    confusion_matrix
)

from catboost import CatBoostClassifier

pd.set_option("display.max_columns", None)

print("✓ Environment ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.1 MB/s eta 0:00:00
✓ Environment ready


In [2]:
# ============================================================
# LOAD CLEAN DATASET
# ============================================================

from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print(filename)

Uploaded files:


In [3]:
DATA_PATH = "apy_clean.csv"

apy = pd.read_csv(DATA_PATH)

print("Shape:", apy.shape)

display(apy.head())

Shape: (325282, 9)


,state,district,crop,crop_year,season,area,production,yield,zero_production_flag
0,Andaman and Nicobar Island,NICOBARS,Arecanut,2000,Kharif,1254.0,2000.0,1.594896,0.0
1,Andaman and Nicobar Island,NICOBARS,Arecanut,2001,Kharif,1254.0,2061.0,1.643541,0.0
2,Andaman and Nicobar Island,NICOBARS,Arecanut,2002,Whole Year,1258.0,2083.0,1.655803,0.0
3,Andaman and Nicobar Island,NICOBARS,Arecanut,2003,Whole Year,1261.0,1525.0,1.209358,0.0
4,Andaman and Nicobar Island,NICOBARS,Arecanut,2004,Whole Year,1264.7,806.0,0.637305,0.0


In [7]:
# ============================================================
# MODEL 2 — DATASET VALIDATION
# ============================================================

print("=" * 60)
print("MODEL 2 — DATASET VALIDATION")
print("=" * 60)

print("\nColumns:")
print(apy.columns.tolist())

print("\nShape:")
print(apy.shape)

print("\nMissing values:")
display(apy.isna().sum())

print("\nUnique values:")
print("States   :", apy["state"].nunique())
print("Districts:", apy["district"].nunique())
print("Crops    :", apy["crop"].nunique())
print("Seasons  :", apy["season"].nunique())

print(
    "Years    :",
    apy["crop_year"].min(),
    "-",
    apy["crop_year"].max()
)

print("\nData types:")
print(apy.dtypes)

MODEL 2 — DATASET VALIDATION

Columns:
['state', 'district', 'crop', 'crop_year', 'season', 'area', 'production', 'yield', 'zero_production_flag']

Shape:
(325282, 9)

Missing values:


,0
state,0
district,0
crop,0
crop_year,0
season,0
area,0
production,0
yield,1
zero_production_flag,1



Unique values:
States   : 35
Districts: 685
Crops    : 55
Seasons  : 6
Years    : 1997 - 2019

Data types:
state                    object
district                 object
crop                     object
crop_year                 int64
season                   object
area                    float64
production              float64
yield                   float64
zero_production_flag    float64
dtype: object


In [9]:
# ============================================================
# CREATE HISTORICAL BEST-CROP TARGET
# ============================================================

GROUP_COLS = [
    "state",
    "district",
    "season",
    "crop_year"
]

# Remove zero-yield observations from determining
# the best-performing crop.
recommendation_source = apy[
    apy["yield"] > 0
].copy()

print("Rows available for recommendation:")
print(len(recommendation_source))

Rows available for recommendation:
318970


In [11]:
# ============================================================
# MODEL 2 — BUILD HISTORICAL BEST-CROP TARGET
# ============================================================

GROUP_COLS = [
    "state",
    "district",
    "season",
    "crop_year"
]

recommendation_source = apy[
    apy["production"] > 0
].copy()

print("Original rows:", len(apy))
print("Positive-production rows:", len(recommendation_source))

best_crop_idx = (
    recommendation_source
    .groupby(GROUP_COLS)["yield"]
    .idxmax()
)

best_crop_data = (
    recommendation_source
    .loc[best_crop_idx]
    .copy()
)

best_crop_data = best_crop_data[
    GROUP_COLS + ["crop", "yield"]
].reset_index(drop=True)

best_crop_data.rename(
    columns={"yield": "best_historical_yield"},
    inplace=True
)

print("\nBest-crop dataset created.")
print("Shape:", best_crop_data.shape)

display(best_crop_data.head(10))

Original rows: 325282
Positive-production rows: 318971

Best-crop dataset created.
Shape: (46797, 6)


,state,district,season,crop_year,crop,best_historical_yield
0,Andaman and Nicobar Island,NICOBARS,Autumn,2008,Sugarcane,19.316239
1,Andaman and Nicobar Island,NICOBARS,Autumn,2009,Sugarcane,14.247788
2,Andaman and Nicobar Island,NICOBARS,Autumn,2010,Sugarcane,3.134328
3,Andaman and Nicobar Island,NICOBARS,Autumn,2011,Sugarcane,2.444444
4,Andaman and Nicobar Island,NICOBARS,Autumn,2012,Sugarcane,0.606061
5,Andaman and Nicobar Island,NICOBARS,Autumn,2013,Sugarcane,33.909091
6,Andaman and Nicobar Island,NICOBARS,Autumn,2014,Sugarcane,11.441441
7,Andaman and Nicobar Island,NICOBARS,Autumn,2015,Rice,2.200000
8,Andaman and Nicobar Island,NICOBARS,Autumn,2016,Rice,1.499250
9,Andaman and Nicobar Island,NICOBARS,Autumn,2017,Rice,1.000000


In [12]:
# ============================================================
# MODEL 2 — TARGET DISTRIBUTION
# ============================================================

crop_distribution = (
    best_crop_data["crop"]
    .value_counts()
    .sort_values(ascending=False)
)

print(
    "Number of unique recommendation classes:",
    best_crop_data["crop"].nunique()
)

display(
    crop_distribution.to_frame("count")
)

Number of unique recommendation classes: 51


,count
crop,
Rice,8200
Sugarcane,7366
Maize,5695
Wheat,4835
Potato,4040
Onion,3567
Coconut,2460
Cotton(lint),1213
Banana,1118


In [13]:
# ============================================================
# MODEL 2 — HISTORICAL CROP PERFORMANCE FEATURES
# ============================================================

# We calculate historical performance for every
# district + season + crop combination.

historical_crop_stats = (
    apy[apy["production"] > 0]
    .groupby(["state", "district", "season", "crop"])
    .agg(
        mean_yield=("yield", "mean"),
        median_yield=("yield", "median"),
        max_yield=("yield", "max"),
        min_yield=("yield", "min"),
        yield_std=("yield", "std"),
        mean_area=("area", "mean"),
        years_cultivated=("crop_year", "nunique")
    )
    .reset_index()
)

# Replace NaN std for crops with only one observation
historical_crop_stats["yield_std"] = (
    historical_crop_stats["yield_std"]
    .fillna(0)
)

# Coefficient of variation
historical_crop_stats["yield_cv"] = (
    historical_crop_stats["yield_std"] /
    historical_crop_stats["mean_yield"].replace(0, np.nan)
)

historical_crop_stats["yield_cv"] = (
    historical_crop_stats["yield_cv"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("Historical crop statistics shape:")
print(historical_crop_stats.shape)

print("\nColumns:")
print(historical_crop_stats.columns.tolist())

display(
    historical_crop_stats.head(20)
)

Historical crop statistics shape:
(32400, 12)

Columns:
['state', 'district', 'season', 'crop', 'mean_yield', 'median_yield', 'max_yield', 'min_yield', 'yield_std', 'mean_area', 'years_cultivated', 'yield_cv']


,state,district,season,crop,mean_yield,median_yield,max_yield,min_yield,yield_std,mean_area,years_cultivated,yield_cv
0,Andaman and Nicobar Island,NICOBARS,Autumn,Arecanut,0.744573,0.744573,0.751264,0.737883,0.009462,4150.000000,2,0.012708
1,Andaman and Nicobar Island,NICOBARS,Autumn,Banana,9.507578,9.507578,9.698113,9.317043,0.269458,796.500000,2,0.028341
2,Andaman and Nicobar Island,NICOBARS,Autumn,Black pepper,0.054558,0.054558,0.068333,0.040783,0.019481,606.500000,2,0.357068
3,Andaman and Nicobar Island,NICOBARS,Autumn,Rice,1.897223,2.000000,2.857143,1.000000,0.542550,1461.101818,11,0.285970
4,Andaman and Nicobar Island,NICOBARS,Autumn,Sugarcane,8.059707,3.233831,33.909091,0.606061,10.125517,27.985833,12,1.256313
5,Andaman and Nicobar Island,NICOBARS,Autumn,Sweet potato,5.538462,5.538462,5.538462,5.538462,0.000000,65.000000,1,0.000000
6,Andaman and Nicobar Island,NICOBARS,Autumn,Tapioca,7.851852,7.851852,7.851852,7.851852,0.000000,108.000000,1,0.000000
7,Andaman and Nicobar Island,NICOBARS,Kharif,Arecanut,1.546086,1.594896,1.643541,1.399820,0.128984,1649.200000,3,0.083426
8,Andaman and Nicobar Island,NICOBARS,Kharif,Banana,8.891939,8.891939,8.891939,8.891939,0.000000,837.950000,1,0.000000
9,Andaman and Nicobar Island,NICOBARS,Kharif,Black pepper,0.057352,0.057352,0.057352,0.057352,0.000000,366.160000,1,0.000000


In [14]:
# ============================================================
# MODEL 2 — SANITY CHECK
# ============================================================

print("Unique states:",
      historical_crop_stats["state"].nunique())

print("Unique districts:",
      historical_crop_stats["district"].nunique())

print("Unique crops:",
      historical_crop_stats["crop"].nunique())

print("Unique seasons:",
      historical_crop_stats["season"].nunique())

print("\nMissing values:")
print(historical_crop_stats.isna().sum())

print("\nYield CV statistics:")
display(
    historical_crop_stats["yield_cv"].describe()
)

Unique states: 35
Unique districts: 685
Unique crops: 55
Unique seasons: 6

Missing values:
state               0
district            0
season              0
crop                0
mean_yield          0
median_yield        0
max_yield           0
min_yield           0
yield_std           0
mean_area           0
years_cultivated    0
yield_cv            0
dtype: int64

Yield CV statistics:


,yield_cv
count,32400.000000
mean,0.251251
std,0.253596
min,0.000000
25%,0.062717
50%,0.211622
75%,0.358234
max,3.751642


In [15]:
# ============================================================
# MODEL 2 — NORMALIZE HISTORICAL PERFORMANCE
# ============================================================

from sklearn.preprocessing import MinMaxScaler

# Work on a copy
recommendation_data = historical_crop_stats.copy()

# ------------------------------------------------------------
# 1. Yield score
# Higher historical yield = better
# ------------------------------------------------------------

yield_scaler = MinMaxScaler()

recommendation_data["yield_score"] = (
    yield_scaler.fit_transform(
        recommendation_data[["median_yield"]]
    ).ravel()
)

# ------------------------------------------------------------
# 2. Stability score
# Lower coefficient of variation = more stable
# ------------------------------------------------------------

stability_scaler = MinMaxScaler()

recommendation_data["cv_scaled"] = (
    stability_scaler.fit_transform(
        recommendation_data[["yield_cv"]]
    ).ravel()
)

recommendation_data["stability_score"] = (
    1 - recommendation_data["cv_scaled"]
)

# ------------------------------------------------------------
# 3. Historical experience score
# More years cultivated = stronger historical evidence
# ------------------------------------------------------------

experience_scaler = MinMaxScaler()

recommendation_data["experience_score"] = (
    experience_scaler.fit_transform(
        recommendation_data[["years_cultivated"]]
    ).ravel()
)

# ------------------------------------------------------------
# 4. Historical recommendation score
#
# Yield       = 50%
# Stability   = 30%
# Experience  = 20%
# ------------------------------------------------------------

recommendation_data["historical_score"] = (
    0.50 * recommendation_data["yield_score"]
    + 0.30 * recommendation_data["stability_score"]
    + 0.20 * recommendation_data["experience_score"]
)

print("Recommendation data shape:")
print(recommendation_data.shape)

display(
    recommendation_data[
        [
            "state",
            "district",
            "season",
            "crop",
            "median_yield",
            "yield_cv",
            "years_cultivated",
            "yield_score",
            "stability_score",
            "experience_score",
            "historical_score"
        ]
    ].head(20)
)

Recommendation data shape:
(32400, 17)


,state,district,season,crop,median_yield,yield_cv,years_cultivated,yield_score,stability_score,experience_score,historical_score
0,Andaman and Nicobar Island,NICOBARS,Autumn,Arecanut,0.744573,0.012708,2,0.000040,0.996613,0.045455,0.308095
1,Andaman and Nicobar Island,NICOBARS,Autumn,Banana,9.507578,0.028341,2,0.000509,0.992446,0.045455,0.307079
2,Andaman and Nicobar Island,NICOBARS,Autumn,Black pepper,0.054558,0.357068,2,0.000003,0.904823,0.045455,0.280539
3,Andaman and Nicobar Island,NICOBARS,Autumn,Rice,2.000000,0.285970,11,0.000107,0.923775,0.454545,0.368095
4,Andaman and Nicobar Island,NICOBARS,Autumn,Sugarcane,3.233831,1.256313,12,0.000173,0.665130,0.500000,0.299625
5,Andaman and Nicobar Island,NICOBARS,Autumn,Sweet potato,5.538462,0.000000,1,0.000296,1.000000,0.000000,0.300148
6,Andaman and Nicobar Island,NICOBARS,Autumn,Tapioca,7.851852,0.000000,1,0.000420,1.000000,0.000000,0.300210
7,Andaman and Nicobar Island,NICOBARS,Kharif,Arecanut,1.594896,0.083426,3,0.000085,0.977763,0.090909,0.311553
8,Andaman and Nicobar Island,NICOBARS,Kharif,Banana,8.891939,0.000000,1,0.000476,1.000000,0.000000,0.300238
9,Andaman and Nicobar Island,NICOBARS,Kharif,Black pepper,0.057352,0.000000,1,0.000003,1.000000,0.000000,0.300001


In [16]:
# ============================================================
# MODEL 2 — SCORE SANITY CHECK
# ============================================================

print("Historical score statistics:")
display(
    recommendation_data["historical_score"].describe()
)

print("\nMinimum score:",
      recommendation_data["historical_score"].min())

print("Maximum score:",
      recommendation_data["historical_score"].max())

print("\nScore outside [0,1]:",
      (
          (recommendation_data["historical_score"] < 0) |
          (recommendation_data["historical_score"] > 1)
      ).sum()
)

Historical score statistics:


,historical_score
count,32400.000000
mean,0.361431
std,0.070125
min,0.119090
25%,0.300187
50%,0.336323
75%,0.422950
max,0.845272



Minimum score: 0.11908969977356035
Maximum score: 0.8452717726483091

Score outside [0,1]: 0


In [17]:
# ============================================================
# MODEL 2 — TOP-K CROP RECOMMENDATION ENGINE
# ============================================================

def recommend_crops(state, district, season=None, top_k=5):
    """
    Recommend the best crops for a given location and season.

    Parameters
    ----------
    state : str
        State name.

    district : str
        District name.

    season : str, optional
        Agricultural season.
        If None, recommendations are based on all seasons.

    top_k : int
        Number of crops to recommend.

    Returns
    -------
    pandas.DataFrame
        Ranked crop recommendations.
    """

    data = recommendation_data.copy()

    # --------------------------------------------------------
    # Normalize input
    # --------------------------------------------------------

    state_input = str(state).strip().lower()
    district_input = str(district).strip().lower()

    data["_state_match"] = (
        data["state"].astype(str).str.strip().str.lower()
        == state_input
    )

    data["_district_match"] = (
        data["district"].astype(str).str.strip().str.lower()
        == district_input
    )

    # --------------------------------------------------------
    # Filter location
    # --------------------------------------------------------

    data = data[
        data["_state_match"] &
        data["_district_match"]
    ].copy()

    if data.empty:
        return pd.DataFrame(
            columns=[
                "rank",
                "crop",
                "season",
                "median_yield",
                "yield_cv",
                "years_cultivated",
                "historical_score"
            ]
        )

    # --------------------------------------------------------
    # Optional season filtering
    # --------------------------------------------------------

    if season is not None:

        season_input = str(season).strip().lower()

        season_data = data[
            data["season"]
            .astype(str)
            .str.strip()
            .str.lower()
            == season_input
        ].copy()

        # If requested season has no data,
        # return an empty result rather than silently
        # using another season.
        if not season_data.empty:
            data = season_data

    # --------------------------------------------------------
    # Rank crops
    # --------------------------------------------------------

    data = (
        data
        .sort_values(
            "historical_score",
            ascending=False
        )
        .drop_duplicates(
            subset=["crop"],
            keep="first"
        )
        .head(top_k)
        .copy()
    )

    # --------------------------------------------------------
    # Add ranking
    # --------------------------------------------------------

    data.insert(
        0,
        "rank",
        range(1, len(data) + 1)
    )

    # --------------------------------------------------------
    # Return clean output
    # --------------------------------------------------------

    return data[
        [
            "rank",
            "crop",
            "season",
            "median_yield",
            "yield_cv",
            "years_cultivated",
            "historical_score"
        ]
    ]

In [18]:
# ============================================================
# MODEL 2 — TEST RECOMMENDATION ENGINE
# ============================================================

recommendations = recommend_crops(
    state="Andaman and Nicobar Island",
    district="NICOBARS",
    top_k=5
)

display(recommendations)

,rank,crop,season,median_yield,yield_cv,years_cultivated,historical_score
43,1,Coconut,Whole Year,4456.110852,0.341893,20,0.564666
48,2,Tapioca,Whole Year,8.453552,0.319586,12,0.374671
3,3,Rice,Autumn,2.000000,0.285970,11,0.368095
40,4,Banana,Whole Year,5.885441,1.004724,16,0.356178
19,5,Dry chillies,Rabi,1.142857,0.587098,11,0.343992


In [19]:
recommendations_kharif = recommend_crops(
    state="Andaman and Nicobar Island",
    district="NICOBARS",
    season="Kharif",
    top_k=5
)

display(recommendations_kharif)

,rank,crop,season,median_yield,yield_cv,years_cultivated,historical_score
12,1,Rice,Kharif,2.981285,0.469347,7,0.317094
7,2,Arecanut,Kharif,1.594896,0.083426,3,0.311553
11,3,Other Kharif pulses,Kharif,0.500000,0.000000,2,0.309104
13,4,Sugarcane,Kharif,19.990610,0.000000,1,0.300535
8,5,Banana,Kharif,8.891939,0.000000,1,0.300238


In [20]:
# ============================================================
# MODEL 2 — RECOMMENDATION EVALUATION
# ============================================================

def evaluate_recommendations(data, top_k=5):
    """
    Evaluate whether the historical ranking places
    high-yield crops inside the Top-K recommendations.
    """

    evaluation_rows = []

    # Evaluate each location-season combination
    groups = data.groupby(
        ["state", "district", "season"],
        dropna=False
    )

    for (state, district, season), group in groups:

        # Need at least top_k crops
        if len(group) < top_k:
            continue

        # Model recommendation
        predicted = (
            group
            .sort_values(
                "historical_score",
                ascending=False
            )
            .drop_duplicates("crop")
            .head(top_k)
        )

        # Actual best crops according to historical median yield
        actual_best = (
            group
            .sort_values(
                "median_yield",
                ascending=False
            )
            .drop_duplicates("crop")
            .head(top_k)
        )

        predicted_crops = set(predicted["crop"])
        actual_crops = set(actual_best["crop"])

        overlap = len(
            predicted_crops & actual_crops
        )

        evaluation_rows.append({
            "state": state,
            "district": district,
            "season": season,
            "top_k_overlap": overlap,
            "hit": int(overlap > 0)
        })

    evaluation = pd.DataFrame(evaluation_rows)

    return evaluation


model2_evaluation = evaluate_recommendations(
    recommendation_data,
    top_k=5
)

print("Evaluation groups:",
      len(model2_evaluation))

print(
    "\nAverage Top-5 overlap:",
    model2_evaluation["top_k_overlap"].mean()
)

print(
    "Top-5 hit rate:",
    model2_evaluation["hit"].mean()
)

display(
    model2_evaluation.head(20)
)

Evaluation groups: 2189

Average Top-5 overlap: 2.590680676107812
Top-5 hit rate: 0.9328460484239379


,state,district,season,top_k_overlap,hit
0,Andaman and Nicobar Island,NICOBARS,Autumn,4,1
1,Andaman and Nicobar Island,NICOBARS,Kharif,4,1
2,Andaman and Nicobar Island,NICOBARS,Rabi,2,1
3,Andaman and Nicobar Island,NICOBARS,Summer,3,1
4,Andaman and Nicobar Island,NICOBARS,Whole Year,3,1
5,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Rabi,2,1
6,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Summer,5,1
7,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Whole Year,4,1
8,Andaman and Nicobar Island,SOUTH ANDAMANS,Rabi,2,1
9,Andaman and Nicobar Island,SOUTH ANDAMANS,Whole Year,3,1


In [21]:
# ============================================================
# MODEL 2 — DETAILED EVALUATION
# ============================================================

avg_overlap = model2_evaluation["top_k_overlap"].mean()

hit_rate = model2_evaluation["hit"].mean()

exact_top5_rate = (
    model2_evaluation["top_k_overlap"] == 5
).mean()

zero_hit_rate = (
    model2_evaluation["top_k_overlap"] == 0
).mean()

print("============================================================")
print("MODEL 2 — FINAL METRICS")
print("============================================================")

print(f"Evaluation groups       : {len(model2_evaluation):,}")
print(f"Average Top-5 overlap   : {avg_overlap:.4f}")
print(f"Top-5 historical hit   : {hit_rate * 100:.2f}%")
print(f"Exact Top-5 overlap     : {exact_top5_rate * 100:.2f}%")
print(f"Zero-hit rate           : {zero_hit_rate * 100:.2f}%")

MODEL 2 — FINAL METRICS
Evaluation groups       : 2,189
Average Top-5 overlap   : 2.5907
Top-5 historical hit   : 93.28%
Exact Top-5 overlap     : 8.59%
Zero-hit rate           : 6.72%


In [22]:
# ============================================================
# MODEL 2 — FINAL INFERENCE WRAPPER
# ============================================================

MODEL_2_VERSION = "1.0"

def model_2_predict(
    state,
    district,
    season=None,
    top_k=5
):
    """
    Backend-ready Model 2 inference interface.

    Input:
        state
        district
        season
        top_k

    Output:
        Ranked crop recommendations.
    """

    result = recommend_crops(
        state=state,
        district=district,
        season=season,
        top_k=top_k
    )

    if result.empty:
        return {
            "success": False,
            "model": "crop_recommendation",
            "version": MODEL_2_VERSION,
            "message": "No historical crop data found for the requested location/season.",
            "recommendations": []
        }

    recommendations = []

    for _, row in result.iterrows():

        recommendations.append({
            "rank": int(row["rank"]),
            "crop": row["crop"],
            "season": row["season"],
            "median_yield": float(row["median_yield"]),
            "yield_cv": float(row["yield_cv"]),
            "years_cultivated": int(row["years_cultivated"]),
            "historical_score": float(row["historical_score"])
        })

    return {
        "success": True,
        "model": "crop_recommendation",
        "version": MODEL_2_VERSION,
        "state": state,
        "district": district,
        "season": season,
        "recommendations": recommendations
    }

In [23]:
# ============================================================
# MODEL 2 — BACKEND INTERFACE TEST
# ============================================================

import json

result = model_2_predict(
    state="Andaman and Nicobar Island",
    district="NICOBARS",
    season="Kharif",
    top_k=5
)

print(
    json.dumps(
        result,
        indent=2
    )
)

{
  "success": true,
  "model": "crop_recommendation",
  "version": "1.0",
  "state": "Andaman and Nicobar Island",
  "district": "NICOBARS",
  "season": "Kharif",
  "recommendations": [
    {
      "rank": 1,
      "crop": "Rice",
      "season": "Kharif     ",
      "median_yield": 2.9812851542525998,
      "yield_cv": 0.46934656716117895,
      "years_cultivated": 7,
      "historical_score": 0.31709389327539117
    },
    {
      "rank": 2,
      "crop": "Arecanut",
      "season": "Kharif     ",
      "median_yield": 1.594896331738437,
      "yield_cv": 0.08342612756496878,
      "years_cultivated": 3,
      "historical_score": 0.31155326779546505
    },
    {
      "rank": 3,
      "crop": "Other Kharif pulses",
      "season": "Kharif     ",
      "median_yield": 0.5,
      "yield_cv": 0.0,
      "years_cultivated": 2,
      "historical_score": 0.3091042200114647
    },
    {
      "rank": 4,
      "crop": "Sugarcane",
      "season": "Kharif     ",
      "median_yield": 19.9906

In [24]:
# ============================================================
# MODEL 2 — SAVE ARTIFACTS
# ============================================================

import os
import joblib

MODEL_2_DIR = "/content/model_2_artifacts"

os.makedirs(
    MODEL_2_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Save recommendation dataset
# ------------------------------------------------------------

recommendation_data.to_csv(
    f"{MODEL_2_DIR}/model_2_recommendation_data.csv",
    index=False
)

# ------------------------------------------------------------
# Save scalers
# ------------------------------------------------------------

joblib.dump(
    yield_scaler,
    f"{MODEL_2_DIR}/yield_scaler.pkl"
)

joblib.dump(
    stability_scaler,
    f"{MODEL_2_DIR}/stability_scaler.pkl"
)

joblib.dump(
    experience_scaler,
    f"{MODEL_2_DIR}/experience_scaler.pkl"
)

# ------------------------------------------------------------
# Save configuration
# ------------------------------------------------------------

model_2_config = {
    "model_name": "Crop Recommendation Engine",
    "model_version": MODEL_2_VERSION,
    "score_components": {
        "yield_score": 0.50,
        "stability_score": 0.30,
        "experience_score": 0.20
    },
    "default_top_k": 5,
    "features": [
        "state",
        "district",
        "season",
        "crop"
    ],
    "historical_features": [
        "median_yield",
        "yield_cv",
        "years_cultivated"
    ],
    "target": "historical_score"
}

joblib.dump(
    model_2_config,
    f"{MODEL_2_DIR}/model_2_config.pkl"
)

print("============================================================")
print("MODEL 2 ARTIFACTS SAVED")
print("============================================================")

for filename in os.listdir(MODEL_2_DIR):
    print(filename)

MODEL 2 ARTIFACTS SAVED
experience_scaler.pkl
model_2_config.pkl
stability_scaler.pkl
yield_scaler.pkl
model_2_recommendation_data.csv


In [25]:
# ============================================================
# ZIP MODEL 2 ARTIFACTS
# ============================================================

import shutil

zip_path = shutil.make_archive(
    "/content/model_2_artifacts",
    "zip",
    MODEL_2_DIR
)

print("Created:")
print(zip_path)

Created:
/content/model_2_artifacts.zip
